# Quantize the own model

Exports run04's `best.pt` to ONNX (FP32), quantizes it to INT8 (post-training static
quantization through ONNX Runtime), and scores the versions with the same mAP code as
everything else.

**A CPU runtime is enough, and costs no compute units** -- this is inference only, and a
CPU is what the model is being optimized for anyway.

- `fp32`: the exported model, the reference. It should reproduce run04's val mAP@0.5 of
  0.4016 -- if not, stop: something's wrong before quantization even started.
- `int8`: everything quantized (`quantize_model.py`), calibrated on 200 training images.
- `int8 float heads`: the same, but the final box/obj/cls convs stay FP32.

Latency isn't measured here: a shared Colab CPU isn't the target. Download the models
at the end and run `benchmark.py` on the machine you care about.

## 1. Settings + mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/yolo-edge-quantization"
RUN_NAME = "run04_long"
SPLIT = "valid"   # pick the variant on "valid"; "test" only for the final numbers
IMG_SIZE = 320
NMS_IOU = 0.5     # tuned for this model on val, see results/comparison

runDir = f"{RUN_ROOT}/{RUN_NAME}"
quantDir = f"{runDir}/quantized"

## 2. Download the dataset

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")

## 3. Clone (or update) the repo

In [ ]:
import os
import sys

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

!pip install -q -r {repoDirectory}/requirements.txt
!git -C {repoDirectory} log --oneline -1

sys.path.insert(0, repoDirectory)

## 4. Export + quantize

`export_model.py` checks that the ONNX model's outputs match PyTorch's. Models are saved
to `runDir/quantized` on Drive.

In [ ]:
import json

os.makedirs(quantDir, exist_ok=True)
manifestPath = f"{runDir}/manifest.json"
with open(manifestPath) as f:
    manifest = json.load(f)
with open(f"{runDir}/classes.json") as f:
    classNames = json.load(f)
numClasses = len(classNames)

MODELS = {
    "fp32": f"{quantDir}/model_fp32.onnx",
    "int8": f"{quantDir}/model_int8.onnx",
    "int8 float heads": f"{quantDir}/model_int8_float_heads.onnx",
}

!python {repoDirectory}/export_model.py --checkpoint {runDir}/best.pt --num-classes {numClasses} --img-size {IMG_SIZE} --out "{MODELS['fp32']}"
!python {repoDirectory}/quantize_model.py --onnx "{MODELS['fp32']}" --out "{MODELS['int8']}" --manifest {manifestPath} --img-size {IMG_SIZE}
!python {repoDirectory}/quantize_model.py --onnx "{MODELS['fp32']}" --out "{MODELS['int8 float heads']}" --manifest {manifestPath} --img-size {IMG_SIZE} --float-heads

for name, path in MODELS.items():
    print(f"{name:18s} {os.path.getsize(path) / 1e6:6.2f} MB")

## 5. Accuracy

Same protocol as `compare_models.ipynb`: conf 0.001 for mAP, class-agnostic mAP to
separate "found the object" from "named it correctly", and each version's own best-F1
operating point.

In [ ]:
import onnxruntime as ort
import pandas as pd
import torch

from train.analysis import best_f1, manifest_gts, per_class_ap, predict_onnx
from train.metrics import mean_average_precision

if SPLIT == "test":
    print("*** TEST SPLIT -- final numbers only; make no decisions from these results ***")

samples = manifest[SPLIT]
gts = manifest_gts(samples)


def agnostic(preds, gts):
    return ([(b, s, torch.zeros_like(l)) for b, s, l in preds], [(b, torch.zeros_like(l)) for b, l in gts])


rows, predsByName = [], {}
for name, path in MODELS.items():
    session = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    preds = predict_onnx(session, samples, IMG_SIZE, conf_thresh=0.001, nms_iou=NMS_IOU)
    predsByName[name] = preds

    op, _ = best_f1(preds, gts)
    rows.append({
        "model": name, "size (MB)": os.path.getsize(path) / 1e6,
        "mAP@0.5": mean_average_precision(preds, gts, numClasses)[0.5],
        "class-agnostic mAP@0.5": mean_average_precision(*agnostic(preds, gts), 1)[0.5],
        "best-F1 conf": op["conf"], "precision": op["precision"], "recall": op["recall"], "F1": op["f1"],
    })

accuracy = pd.DataFrame(rows).set_index("model")
with open(f"{quantDir}/accuracy_{SPLIT}.json", "w") as f:
    json.dump(rows, f, indent=1)

if SPLIT == "valid" and abs(accuracy.loc["fp32", "mAP@0.5"] - 0.4016) > 0.002:
    print("WARNING: fp32 doesn't reproduce run04's val mAP@0.5 of 0.4016 -- check before trusting the rest")
accuracy.round(4).T

## 6. Which classes lost the most

Mean AP is one number; this shows whether quantization hurt everywhere or a few classes
(against `fp32`, for the classes with the biggest drop).

In [ ]:
perClass = pd.DataFrame({"class": classNames})
for name, preds in predsByName.items():
    perClass[name] = per_class_ap(preds, gts, numClasses)
perClass["int8 - fp32"] = perClass["int8"] - perClass["fp32"]
perClass.dropna().sort_values("int8 - fp32").head(10).round(3).reset_index(drop=True)

## 7. Latency, on your own machine

Download the models (or take them from `runDir/quantized` on Drive), then in the repo:

```
python benchmark.py fp32=model_fp32.onnx int8=model_int8.onnx "int8_fh=model_int8_float_heads.onnx" --threads 1 2 4
```

In [ ]:
from google.colab import files

for path in MODELS.values():
    files.download(path)